# NexRoute | Logistics & Delivery Performance
John Paul Torres · fictional Philippine logistics simulation, January 2024–December 2025.

Execute from the repository root. The local SQLite file is generated using `python scripts/generate_data.py` then `python scripts/load_database.py`.

In [ ]:
from pathlib import Path
import sys,sqlite3
ROOT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
import pandas as pd, numpy as np, geopandas as gpd
import seaborn as sns, matplotlib.pyplot as plt
import plotly.express as px
from src.database import read_shipments, DB_PATH
from src.data_processing import prepare_shipments
from src.delivery_metrics import overview,grouped_performance
from src.sla_analysis import sla_summary,pareto
from src.geospatial_analysis import destination_points
s=prepare_shipments(read_shipments())
s.shape

## 1. Quality & completeness checks
Valid keys, statuses, dispatch ordering and null delivery dates.

In [ ]:
assert s.Shipment_ID.is_unique
assert s.Dispatch_Date.ge(s.Shipment_Date).all()
assert s.loc[s.Delivered,'Actual_Delivery_Date'].notna().all()
s.Delivery_Status.value_counts()

## 2. Relational SQL join

In [ ]:
with sqlite3.connect(DB_PATH) as c:
    sql=pd.read_sql_query('SELECT s.Shipment_ID,c.Courier_Name,h.Hub_Name FROM shipments s JOIN couriers c ON s.Courier_ID=c.Courier_ID JOIN hubs h ON s.Origin_Hub_ID=h.Hub_ID LIMIT 12',c)
sql

## 3. KPI calculations
Success = delivered / (delivered + failed + returned); on-time denominator is delivered only.

In [ ]:
overview(s)

## 4. Shipment volume and trends

In [ ]:
volume=s.groupby('Month').size().rename('Shipments').reset_index()
px.line(volume,x='Month',y='Shipments',title='Synthetic monthly shipment volume').show()

## 5. Courier performance and SLA compliance

In [ ]:
courier=grouped_performance(s,['Courier_ID','Courier_Name','Assigned_Region'])
courier[courier.delivered>=75].nlargest(15,'on_time_rate')[['Courier_Name','delivered','on_time_rate','success_rate']]

In [ ]:
sla_summary(s)


In [ ]:
pareto(s).head(10)

## 6. Cost correlations and delivery time distribution
Modeled distance and cost relationships reflect the simulation formula, not real-world causality.

In [ ]:
numeric=['Distance_KM','Package_Weight_KG','Shipping_Cost_PHP','Delivery_Days']
fig,ax=plt.subplots(figsize=(8,4))
sns.heatmap(s[numeric].corr(),annot=True,ax=ax)
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
sns.histplot(s.loc[s.Delivered,'Delivery_Days'],bins=20,color='steelblue',ax=ax)
plt.show()

## 7. Geographic aggregation
City-center coordinates are representative points and not customer addresses.

In [ ]:
geo=destination_points(s)
geo[['Destination_Region','Destination_City','shipments','on_time_rate','geometry']].head()

In [ ]:
fig=px.scatter_geo(geo,lat='Destination_Latitude',lon='Destination_Longitude',size='shipments',color='on_time_rate',hover_name='Destination_City',scope='asia',projection='mercator')
fig.update_geos(lataxis_range=[4,21],lonaxis_range=[116,127]);fig.show()

## 8. Recommendations and limitations
Review lowest-performing regional SLA groups; compare courier rankings only above volume thresholds; investigate reasons for failed delivery. All data and costs are fictional. Distance is WGS84 geodesic, not actual road distance. No official administrative boundaries or route engines are used.